### Imports


In [1]:
# Imports - basic
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

# Configs
from config.config import SAE_Config
from config.config import CNN_Config

# Classes
from dataloader.load_data import LoadData
from model.cnn import ExplainableCNN
from model.sae import SAE
from trainer.trainer import Trainer

# Utils
from utils.hooks import Hooks

### Dataset and transforms


In [2]:
# Define transforms
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

# Load Dataset
data = LoadData(transform=transform)
training_data, testing_data = data.load_data("CIFAR10")

# Data Loaders
train_loader = data.data_loaders(training_data, "train")
test_loader = data.data_loaders(testing_data, "test")

# Example visualization
examples = iter(train_loader)
samples, labels = next(examples)
print(samples.shape, labels.shape)

Files already downloaded and verified
Files already downloaded and verified
torch.Size([10, 3, 32, 32]) torch.Size([10])


### Model Configs


In [3]:
# Import configs
sae_model_arguments = SAE_Config['model_args']
sae_model_arguments

{'input_dims': 10,
 'hidden_dims': 25,
 'epochs': 2,
 'batch_size': 1,
 'learning_rate': 0.01}

### Model


In [4]:
sae = SAE(sae_model_arguments['input_dims'], nn.ReLU, sae_model_arguments['hidden_dims'])
sae

SAE(
  (encoder): Linear(in_features=10, out_features=25, bias=True)
  (relu): ReLU()
  (decoder): Linear(in_features=25, out_features=10, bias=True)
)

In [5]:
# Define the criterion and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(sae.parameters(), lr=sae_model_arguments['learning_rate'])

### Feature extraction


In [6]:
experiment_1_configs = CNN_Config['model_args']
experiment_1_configs

{'input_size': 3,
 'num_classes': 10,
 'hidden_layers': [64, 128, 256, 128, 64],
 'activation': torch.nn.modules.activation.ReLU,
 'norm_layer': torch.nn.modules.batchnorm.BatchNorm2d,
 'drop_prob': 0.4,
 'max_pool': False}

In [7]:
cnn_model = ExplainableCNN(input_size=experiment_1_configs['input_size'], hidden_layers=experiment_1_configs['hidden_layers'], 
                       num_classes=experiment_1_configs['num_classes'], activation=experiment_1_configs['activation'], 
                       normalization=experiment_1_configs['norm_layer'], max_pool=experiment_1_configs['max_pool'], 
                       drop_prob=experiment_1_configs['drop_prob'])

cnn_model.load_state_dict(torch.load('model_weights.pth'))
cnn_model.eval()

/var/folders/lg/dkxq14195c770j5_whjgkvj80000gn/T/ipykernel_5099/1193232790.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cnn_model.load_state_dict(torch.load('model_we

ExplainableCNN(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (norm3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv4): Conv2d(256, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (norm4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv5): Conv2d(128, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (norm5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear): Linear(in_features=65536, out_features=10, bias=True)
  (relu): ReLU()
)

In [8]:
feature_collector = Hooks(cnn_model, ['conv5'])

In [9]:
for i in range(5):
    data, _ = testing_data[i]
    data.unsqueeze_(0)
    output = cnn_model(data)
    feature_collector.collect_features('conv5')

len(feature_collector.feature_vectors)

5120

### Training


In [10]:
# Training
trainer = LoadData(transform=None)
trainer.create_sae_loaders(features=feature_collector.feature_vectors, batch_size=20)
sae_train_loader = trainer.train_loader
sae_test_loader = trainer.test_loader

In [ ]:
# Saving

204
51


### Hooks


In [ ]:
# Hooks on the SAE
sae_model = SAE(sae_model_arguments['input_dims'], nn.ReLU, sae_model_arguments['hidden_dims'])

collector = Hooks(model=sae_model, module_names=['encoder'])
activations = collector.activations['encoder']
